# Dataframes in Python — Problems and Solutions
**Technical Notes**  
*Rodrigo Kang*

This notebook contains the solutions to the problems included in the associated technical notes on DataFrames in Python. The emphasis is on structural reasoning as well as pandas syntax: labels, types, missingness, grouping, alignment, merge cardinality, grain, relational data, and preparation for machine learning.

In [ ]:
import numpy as np
import pandas as pd

## Problem 1 — Creating a DataFrame

### Problem

Create the supplied `name`, `age`, and `income` data as a DataFrame and inspect its structure.

### Solution

Dictionary keys become column labels and equally sized sequences become column values. Each row is one person; each column is one variable.

#### Implementation

In [ ]:
data = {
    "name": ["Alice", "Bob", "Carol", "David"],
    "age": [25, 31, 28, 40],
    "income": [42000, 51000, 47000, 68000]
}
df = pd.DataFrame(data)
print(type(df))
print("shape:", df.shape)
print("ndim:", df.ndim)
print("columns:", df.columns.tolist())
print("index:", df.index)
print(df.dtypes)
df

### Key Takeaway

A DataFrame is a two-dimensional labelled table; always begin by identifying the row grain and column semantics.

## Problem 2 — Series Versus DataFrame

### Problem

Compare `df['age']` with `df[['age']]`.

### Solution

The first is a one-dimensional `Series`; the second is a two-dimensional one-column `DataFrame`.

#### Implementation

In [ ]:
age_series = df["age"]
age_frame = df[["age"]]
print(type(age_series), age_series.ndim, age_series.shape)
print(type(age_frame), age_frame.ndim, age_frame.shape)

### Key Takeaway

Same values do not imply the same dimensional interface.

## Problem 3 — Structural Inspection

### Problem

Create mixed-type columns and inspect them with `head`, `tail`, `info`, `describe`, and `dtypes`.

### Solution

`head`/`tail` inspect rows, `info` summarises schema and null counts, `describe` summarises numerical columns by default, and `dtypes` reports column types.

#### Implementation

In [ ]:
inspect_df = pd.DataFrame({
    "count": [1,2,3,4],
    "value": [1.5,2.2,3.1,4.8],
    "group": ["A","B","A","B"],
    "active": [True,False,True,True]
})
print(inspect_df.head())
print(inspect_df.tail())
inspect_df.info()
print(inspect_df.describe())
print(inspect_df.dtypes)

### Key Takeaway

`shape`, `head`, `info`, `dtypes`, and missing-value counts form a strong first-pass audit.

## Problem 4 — Label-Based and Position-Based Selection

### Problem

Compare `.loc[]` and `.iloc[]` on an integer-labelled index.

### Solution

`.loc` uses labels; `.iloc` uses zero-based positions. Label `20` and position `1` happen to identify the same row here.

#### Implementation

In [ ]:
df4 = pd.DataFrame(
    {"name":["Alice","Bob","Carol","David"], "score":[82,91,76,88]},
    index=[10,20,30,40]
)
print(df4.loc[20])
print(df4.iloc[1])
print(df4.loc[20, "score"])
print(df4.iloc[1, 1])
try:
    df4.loc[1]
except KeyError as e:
    print("df4.loc[1] -> KeyError")

### Key Takeaway

Use `.loc` for labels and `.iloc` for positions.

## Problem 5 — Slicing with `.loc[]` and `.iloc[]`

### Problem

Compare `df.loc[10:30]` and `df.iloc[0:3]`.

### Solution

Label slices generally include the final label; positional slices exclude the stop position. Both select the first three rows here.

#### Implementation

In [ ]:
print(df4.loc[10:30])
print(df4.iloc[0:3])

### Key Takeaway

Label slicing and positional slicing have different endpoint semantics.

## Problem 6 — Boolean Filtering

### Problem

Select rows with `age > 30` and `income < 65000`, returning selected columns.

### Solution

Build Boolean masks and combine them element-wise with `&`, then use `.loc` for simultaneous row and column selection.

#### Implementation

In [ ]:
df6 = pd.DataFrame({
    "name":["Alice","Bob","Carol","David","Eva"],
    "age":[25,31,28,40,36],
    "income":[42000,51000,47000,68000,59000],
    "group":["A","B","A","B","A"]
})
df6.loc[
    (df6["age"] > 30) & (df6["income"] < 65000),
    ["name","age","income"]
]

### Key Takeaway

Filtering defines the analytical population retained.

## Problem 7 — Combining Boolean Conditions

### Problem

Use OR and AND conditions on the same DataFrame.

### Solution

Pandas comparisons produce Boolean Series, so `&` and `|` combine them element-wise. Python's `and`/`or` expect scalar truth values.

#### Implementation

In [ ]:
print(df6.loc[(df6["group"] == "A") | (df6["income"] > 60000)])
print(df6.loc[(df6["group"] == "A") & (df6["age"] >= 30)])

### Key Takeaway

Parenthesise comparisons and use element-wise Boolean operators.

## Problem 8 — Membership Filtering

### Problem

Use `.isin()` and its negation.

### Solution

`.isin()` directly expresses set membership; `~` negates the resulting Boolean mask.

#### Implementation

In [ ]:
names = ["Alice","Carol","Eva"]
print(df6.loc[df6["name"].isin(names)])
print(df6.loc[~df6["name"].isin(names)])

### Key Takeaway

`.isin()` is clearer than chaining many equality tests.

## Problem 9 — Creating a Derived Numerical Column

### Problem

Create `revenue = price * quantity` without a loop.

### Solution

Series arithmetic is vectorised and aligned by index.

#### Implementation

In [ ]:
sales = pd.DataFrame({
    "product":["A","B","C","D"],
    "price":[10.0,15.0,8.0,20.0],
    "quantity":[3,2,5,4]
})
sales["revenue"] = sales["price"] * sales["quantity"]
sales

### Key Takeaway

Vectorised transformations operate at column level while preserving row grain.

## Problem 10 — Conditional Columns

### Problem

Create Boolean and categorical columns from revenue.

### Solution

A vectorised comparison produces a Boolean Series; `np.where` chooses one of two labels per row.

#### Implementation

In [ ]:
sales["high_value"] = sales["revenue"] >= 50
sales["revenue_category"] = np.where(
    sales["revenue"] >= 50, "High", "Standard"
)
sales

### Key Takeaway

Conditional feature construction need not use row iteration.

## Problem 11 — `.assign()` and Method Chaining

### Problem

Calculate revenue, filter, and sort in a chain.

### Solution

Method chaining can make transformation order explicit, though overly long chains can become difficult to debug.

#### Implementation

In [ ]:
sales_base = pd.DataFrame({
    "product":["A","B","C","D"],
    "price":[10.0,15.0,8.0,20.0],
    "quantity":[3,2,5,4]
})
result11 = (
    sales_base
    .assign(revenue=lambda x: x["price"] * x["quantity"])
    .loc[lambda x: x["revenue"] >= 30]
    .sort_values("revenue", ascending=False)
)
result11

### Key Takeaway

Use chaining when it improves readability, not merely compactness.

## Problem 12 — Renaming and Dropping Columns

### Problem

Rename two columns and remove an unused one.

### Solution

`rename` changes labels; `drop(columns=...)` removes variables.

#### Implementation

In [ ]:
df12 = pd.DataFrame({"p":[10,20,30],"q":[2,4,5],"unused":[0,0,0]})
df12.rename(columns={"p":"price","q":"quantity"}).drop(columns="unused")

### Key Takeaway

Column names are part of the DataFrame interface.

## Problem 13 — Type Conversion

### Problem

Convert text amounts to numeric and text dates to datetime.

### Solution

Data types determine which operations have valid semantics. Explicit conversion avoids relying on string behaviour.

#### Implementation

In [ ]:
df13 = pd.DataFrame({
    "id":["1","2","3"],
    "amount":["10.5","7.2","14.8"],
    "date":["2026-01-01","2026-01-02","2026-01-03"]
})
print(df13.dtypes)
df13["amount"] = pd.to_numeric(df13["amount"])
df13["date"] = pd.to_datetime(df13["date"])
print(df13.dtypes)
df13

### Key Takeaway

Values that look numeric or date-like may still be stored as text.

## Problem 14 — Detecting Missing Values

### Problem

Find and count missing values and filter on missing `income`.

### Solution

`.isna()`/`.notna()` create missingness masks; summing Boolean masks counts missing values.

#### Implementation

In [ ]:
df14 = pd.DataFrame({
    "age":[25,np.nan,31,40],
    "income":[42000,51000,np.nan,68000],
    "group":["A","B",None,"B"]
})
print(df14.isna())
print(df14.isna().sum())
print(df14.loc[df14["income"].isna()])
print(df14.loc[df14["income"].notna()])

### Key Takeaway

Detect missingness explicitly before choosing a treatment.

## Problem 15 — Dropping Missing Values

### Problem

Compare `dropna()` with `dropna(subset=['income'])`.

### Solution

The first drops rows missing anywhere; the second uses only `income` to determine retention.

#### Implementation

In [ ]:
print(df14.dropna())
print(df14.dropna(subset=["income"]))

### Key Takeaway

Deletion criteria should follow the analytical requirement, not convenience.

## Problem 16 — Simple Imputation

### Problem

Fill missing income with the observed median.

### Solution

Median imputation is easy to implement but changes the empirical distribution and ignores uncertainty in missing values.

#### Implementation

In [ ]:
df16 = df14.copy()
before = df16["income"].median()
df16["income"] = df16["income"].fillna(before)
after = df16["income"].median()
print(before, after)
df16

### Key Takeaway

Imputation is a modelling decision, not merely a pandas method call.

## Problem 17 — Sorting

### Problem

Sort by income and by multiple columns.

### Solution

Sorting changes row order but not row identity; it matters analytically only when later operations depend on order.

#### Implementation

In [ ]:
print(df6.sort_values("income"))
print(df6.sort_values("income", ascending=False))
print(df6.sort_values(["group","income"], ascending=[True,False]))

### Key Takeaway

Ordering is usually presentation unless sequence carries meaning.

## Problem 18 — Frequency Tables

### Problem

Calculate counts, proportions, cardinality, and unique levels.

### Solution

`value_counts` summarises frequency, `normalize=True` gives proportions, `nunique` gives cardinality, and `unique` lists observed levels.

#### Implementation

In [ ]:
df18 = pd.DataFrame({"country":["NZ","AU","NZ","US","NZ","AU","UK","NZ"]})
print(df18["country"].value_counts())
print(df18["country"].value_counts(normalize=True))
print(df18["country"].nunique())
print(df18["country"].unique())

### Key Takeaway

Frequency and cardinality checks are core categorical diagnostics.

## Problem 19 — String Cleaning

### Problem

Trim and title-case names and cities.

### Solution

The `.str` accessor vectorises ordinary string operations over a Series.

#### Implementation

In [ ]:
df19 = pd.DataFrame({
    "name":["  alice smith","BOB JONES "," Carol Brown "],
    "city":["auckland","WELLINGTON","christchurch"]
})
df19["name"] = df19["name"].str.strip().str.title()
df19["city"] = df19["city"].str.strip().str.title()
df19

### Key Takeaway

Prefer vectorised string accessors to explicit row loops.

## Problem 20 — String Filtering

### Problem

Filter strings by contains, prefix, and suffix.

### Solution

String predicates return Boolean masks. `na=False` treats missing strings as non-matches.

#### Implementation

In [ ]:
text_df = pd.DataFrame({
    "company":["Northwind Traders","Alpha Foods","Southern North Ltd","Beta Ltd",None]
})
print(text_df.loc[text_df["company"].str.contains("north", case=False, na=False)])
print(text_df.loc[text_df["company"].str.startswith("A", na=False)])
print(text_df.loc[text_df["company"].str.endswith("Ltd", na=False)])

### Key Takeaway

Text filters become standard Boolean filtering once expressed through `.str`.

## Problem 21 — Splitting Structured Text

### Problem

Split `NZ-AKL-001`-style codes into variables.

### Solution

`str.split(..., expand=True)` exposes embedded components as explicit columns.

#### Implementation

In [ ]:
df21 = pd.DataFrame({"code":["NZ-AKL-001","NZ-WLG-002","AU-SYD-003"]})
df21[["country","city","id"]] = df21["code"].str.split("-", expand=True)
df21

### Key Takeaway

Explicit variables are easier to validate and analyse than values hidden inside text.

## Problem 22 — Datetime Components

### Problem

Extract calendar components from a datetime column.

### Solution

After `pd.to_datetime`, `.dt` exposes vectorised calendar attributes.

#### Implementation

In [ ]:
df22 = pd.DataFrame({"date":["2026-01-15","2026-03-21","2026-07-04"]})
df22["date"] = pd.to_datetime(df22["date"])
df22["year"] = df22["date"].dt.year
df22["month"] = df22["date"].dt.month
df22["day"] = df22["date"].dt.day
df22["day_name"] = df22["date"].dt.day_name()
df22["quarter"] = df22["date"].dt.quarter
df22

### Key Takeaway

Datetime dtype unlocks calendar-aware transformations.

## Problem 23 — Time Differences

### Problem

Calculate duration between start and end datetimes.

### Solution

Subtracting two datetimes produces a timedelta: an elapsed duration rather than a timestamp.

#### Implementation

In [ ]:
df23 = pd.DataFrame({
    "start":pd.to_datetime(["2026-01-01","2026-02-01"]),
    "end":pd.to_datetime(["2026-01-10","2026-02-20"])
})
df23["duration"] = df23["end"] - df23["start"]
df23["duration_days"] = df23["duration"].dt.days
df23

### Key Takeaway

Datetimes are points in time; timedeltas are differences between them.

## Problem 24 — GroupBy and Mean

### Problem

Compute mean, sum, and count of sales by region.

### Solution

`groupby` splits rows by region, applies each aggregation within groups, and combines group-level results.

#### Implementation

In [ ]:
sales24 = pd.DataFrame({
    "region":["North","North","South","South","South","West"],
    "sales":[100,150,120,180,200,140]
})
print(sales24.groupby("region")["sales"].mean())
print(sales24.groupby("region")["sales"].sum())
print(sales24.groupby("region").size())

### Key Takeaway

Aggregation changes the unit of analysis from rows to groups.

## Problem 25 — Multiple Group Aggregations

### Problem

Create a named department summary.

### Solution

Named aggregation makes the output schema explicit.

#### Implementation

In [ ]:
employees25 = pd.DataFrame({
    "department":["A","A","B","B","B","C"],
    "salary":[60000,65000,70000,72000,68000,75000],
    "experience":[2,4,5,6,3,8]
})
employees25.groupby("department").agg(
    employee_count=("salary","size"),
    mean_salary=("salary","mean"),
    median_salary=("salary","median"),
    mean_experience=("experience","mean")
)

### Key Takeaway

Use output names that state what each statistic means.

## Problem 26 — Aggregation Changes Grain

### Problem

Explain grain before and after grouping employees by department.

### Solution

Before grouping, one row represents one employee. After aggregation, one row represents one department.

### Key Takeaway

Aggregation changes table grain, not just values.

## Problem 27 — Group-Level Transformation

### Problem

Add department mean salary and each employee's deviation from it.

### Solution

`transform` broadcasts a group statistic back to the original index, preserving one row per employee.

#### Implementation

In [ ]:
employees27 = employees25.copy()
employees27["department_mean_salary"] = (
    employees27.groupby("department")["salary"].transform("mean")
)
employees27["salary_difference"] = (
    employees27["salary"] - employees27["department_mean_salary"]
)
employees27

### Key Takeaway

`agg` reduces rows; `transform` preserves them.

## Problem 28 — Concatenating Observations

### Problem

Stack two batches vertically.

### Solution

Both tables have the same schema and represent different observations, so concatenation is appropriate.

#### Implementation

In [ ]:
df_a = pd.DataFrame({"id":[1,2],"value":[10,20]})
df_b = pd.DataFrame({"id":[3,4],"value":[30,40]})
pd.concat([df_a, df_b], ignore_index=True)

### Key Takeaway

Concatenation extends an axis; merging combines rows according to keys.

## Problem 29 — Horizontal Concatenation and Index Alignment

### Problem

Concatenate columns whose indexes are in different orders.

### Solution

Pandas aligns by label before placing columns side by side.

#### Implementation

In [ ]:
left29 = pd.DataFrame({"x":[1,2,3]}, index=["A","B","C"])
right29 = pd.DataFrame({"y":[10,20,30]}, index=["C","B","A"])
pd.concat([left29,right29], axis=1)

### Key Takeaway

Horizontal concatenation is label-aware.

## Problem 30 — Basic Merge

### Problem

Merge orders with customer names.

### Solution

`orders` is the many side and `customers` the one side; the result remains one row per order.

#### Implementation

In [ ]:
customers30 = pd.DataFrame({
    "customer_id":[1,2,3],
    "customer_name":["Alpha","Beta","Gamma"]
})
orders30 = pd.DataFrame({
    "order_id":[101,102,103,104],
    "customer_id":[1,2,1,3],
    "amount":[50,80,30,120]
})
orders30.merge(
    customers30, on="customer_id", how="left", validate="many_to_one"
)

### Key Takeaway

A many-to-one enrichment can preserve the grain of the many-side table.

## Problem 31 — Join Types

### Problem

Compare inner, left, right, and outer joins with unmatched keys.

### Solution

Join type determines which key populations are retained.

#### Implementation

In [ ]:
customers31 = pd.DataFrame({
    "customer_id":[1,2,3,4],
    "customer_name":["Alpha","Beta","Gamma","Delta"]
})
orders31 = pd.DataFrame({
    "order_id":[101,102,103,104,105],
    "customer_id":[1,2,1,3,99],
    "amount":[50,80,30,120,60]
})
for how in ["inner","left","right","outer"]:
    print("\n", how.upper())
    print(orders31.merge(customers31, on="customer_id", how=how))

### Key Takeaway

Join type is a population-retention choice.

## Problem 32 — Cardinality

### Problem

Identify and validate the customer/order relationship.

### Solution

From `orders` to `customers`, many orders map to one customer row, so `many_to_one` is the expected merge cardinality.

#### Implementation

In [ ]:
orders30.merge(
    customers30, on="customer_id", how="left", validate="many_to_one"
)

### Key Takeaway

Merge validation turns an assumption into a checked invariant.

## Problem 33 — Unexpected Row Multiplication

### Problem

Predict a many-to-many merge row count.

### Solution

Two left rows times three right rows gives six pairings for key `A`.

#### Implementation

In [ ]:
left33 = pd.DataFrame({"key":["A","A"],"x":[1,2]})
right33 = pd.DataFrame({"key":["A","A","A"],"y":[10,20,30]})
result33 = left33.merge(right33, on="key")
print(result33)
print(len(result33))

### Key Takeaway

Unexpected many-to-many joins silently inflate row counts.

## Problem 34 — Index Alignment in Series

### Problem

Add same labels in different orders.

### Solution

Pandas matches `A` with `A`, `B` with `B`, and `C` with `C`, not by physical order.

#### Implementation

In [ ]:
a34 = pd.Series([10,20,30], index=["A","B","C"])
b34 = pd.Series([1,2,3], index=["C","B","A"])
a34 + b34

### Key Takeaway

Pandas arithmetic is label-aware.

## Problem 35 — Missing Labels During Alignment

### Problem

Add partially overlapping label sets.

### Solution

Pandas uses the union of labels; missing counterparts produce missing results.

#### Implementation

In [ ]:
a35 = pd.Series([10,20,30], index=["A","B","C"])
b35 = pd.Series([1,2,3], index=["B","C","D"])
a35 + b35

### Key Takeaway

Unexpected NaNs can be an alignment symptom.

## Problem 36 — Duplicate Rows Versus Repeated Keys

### Problem

Differentiate exact duplicates from repeated customer IDs.

### Solution

Repeated customer IDs are legitimate when each row is a different order; duplicate interpretation depends on grain.

#### Implementation

In [ ]:
df36 = pd.DataFrame({
    "customer_id":[1,1,2,2,2],
    "order_id":[101,102,201,201,202],
    "amount":[50,70,40,40,90]
})
print(df36.duplicated())
print(df36.duplicated(subset=["customer_id"]))
print(df36.drop_duplicates())

### Key Takeaway

Duplicate status is meaningful only relative to observational identity.

## Problem 37 — Wide to Long Format

### Problem

Melt yearly sales columns into `customer`, `year`, `sales`.

### Solution

The grain changes from one row per customer to one row per customer-year.

#### Implementation

In [ ]:
wide37 = pd.DataFrame({
    "customer":["A","B","C"],
    "sales_2024":[100,150,120],
    "sales_2025":[130,160,140]
})
long37 = wide37.melt(
    id_vars="customer",
    value_vars=["sales_2024","sales_2025"],
    var_name="year",
    value_name="sales"
)
long37

### Key Takeaway

Reshaping changes representation and often grain without changing the underlying information.

## Problem 38 — Long to Wide Format

### Problem

Reconstruct wide form with `pivot()`.

### Solution

`pivot` requires one unique value for each index-column coordinate because it does not aggregate.

#### Implementation

In [ ]:
long37.pivot(index="customer", columns="year", values="sales")

### Key Takeaway

`pivot` reshapes unique coordinates; `pivot_table` can aggregate repeated ones.

## Problem 39 — Pivot Table with Aggregation

### Problem

Compute mean amount by region and product.

### Solution

Repeated region-product pairs require aggregation, making `pivot_table` appropriate.

#### Implementation

In [ ]:
sales39 = pd.DataFrame({
    "region":["North","North","North","South","South","South"],
    "product":["A","A","B","A","B","B"],
    "amount":[100,120,80,140,110,130]
})
sales39.pivot_table(
    index="region", columns="product", values="amount", aggfunc="mean"
)

### Key Takeaway

A pivot table combines grouping, aggregation, and reshaping.

## Problem 40 — Reading a CSV

### Problem

Write a robust import pattern for `transactions.csv`.

### Solution

Select only required columns, parse dates, then inspect schema and missingness before analysis.

#### Implementation

In [ ]:
csv_solution = """transactions = pd.read_csv(
    "transactions.csv",
    usecols=["customer_id", "transaction_date", "amount", "status"]
)
transactions["transaction_date"] = pd.to_datetime(transactions["transaction_date"])
transactions.info()
print(transactions.isna().sum())
print(transactions.head())
"""
print(csv_solution)

### Key Takeaway

Successful parsing is not equivalent to correct interpretation.

## Problem 41 — Writing a DataFrame

### Problem

Export without the index and explain when index data should be preserved.

### Solution

Use `index=False` when the index is only an internal row label. If it contains meaningful identifiers or time coordinates, preserve or reset it into a column.

#### Implementation

In [ ]:
print('transformed_df.to_csv("processed.csv", index=False)')

### Key Takeaway

Decide whether the index is data before exporting.

## Northwind Setup

The following setup uses the supplied `northwind.sql` file. If it is stored beside this notebook, the cell creates a local SQLite database named `Northwind.db` using the exact schema and data supplied with these notes.

In [ ]:
from pathlib import Path
import sqlite3

sql_path = Path("northwind.sql")
if not sql_path.exists():
    raise FileNotFoundError(
        "Place northwind.sql in the same directory as this notebook before running the Northwind exercises."
    )

db_path = Path("Northwind.db")
connection = sqlite3.connect(db_path)

with sql_path.open("r", encoding="utf-8") as file:
    connection.executescript(file.read())

connection.commit()
print(f"Northwind database created at: {db_path.resolve()}")

## Problem 42 — Northwind Schema Reasoning

### Problem

Identify grain and keys for `Customers`, `Orders`, `OrderDetails`, and `Products`.

### Solution

The supplied schema has one row per customer, order, order detail, and product respectively. `CustomerID`, `OrderID`, and `ProductID` provide the key relationships.

#### Implementation

In [ ]:
pd.read_sql_query("""
SELECT name, sql
FROM sqlite_master
WHERE type = 'table'
  AND name IN ('Customers','Orders','OrderDetails','Products')
ORDER BY name
""", connection)

### Key Takeaway

Relational reasoning begins with grain, keys, and cardinality.

## Problem 43 — Northwind Customers Query

### Problem

Read customer identifiers, names, countries, then count customers by country.

### Solution

SQL retrieves the customer-level table; pandas performs the categorical summary.

#### Implementation

In [ ]:
customers43 = pd.read_sql_query("""
SELECT CustomerID, CustomerName, Country
FROM Customers
""", connection)
print(customers43.head())
customers43["Country"].value_counts()

### Key Takeaway

SQL and pandas can divide responsibilities cleanly.

## Problem 44 — Filtering in SQL Versus pandas

### Problem

Retrieve German customers both ways.

### Solution

The results should match. Filtering in SQL is generally preferable for very large tables because it reduces transfer and memory use.

#### Implementation

In [ ]:
germany_sql = pd.read_sql_query("""
SELECT CustomerID, CustomerName, Country
FROM Customers
WHERE Country = 'Germany'
""", connection)

all_customers44 = pd.read_sql_query("""
SELECT CustomerID, CustomerName, Country
FROM Customers
""", connection)

germany_pandas = all_customers44.loc[
    all_customers44["Country"] == "Germany"
]
print(len(germany_sql), len(germany_pandas))
print(set(germany_sql["CustomerID"]) == set(germany_pandas["CustomerID"]))

### Key Takeaway

Push selective work toward the data source when it materially reduces data movement.

## Problem 45 — Northwind Orders and Customers

### Problem

Join orders to customers and explain grain.

### Solution

Each order has one customer, so customer attributes can be repeated across orders without changing order-level grain.

#### Implementation

In [ ]:
orders45 = pd.read_sql_query("""
SELECT
    o.OrderID,
    o.OrderDate,
    c.CustomerName,
    c.Country
FROM Orders AS o
JOIN Customers AS c
    ON o.CustomerID = c.CustomerID
""", connection)
orders45.head()

### Key Takeaway

Repeated dimension attributes can be expected in one-to-many joins.

## Problem 46 — Northwind Order Lines

### Problem

Join order details to products, compute line value, then total by order.

### Solution

`Quantity × Price` is defined at order-line grain. Grouping by `OrderID` then changes the grain to one row per order.

#### Implementation

In [ ]:
order_lines46 = pd.read_sql_query("""
SELECT
    od.OrderID,
    od.ProductID,
    p.ProductName,
    od.Quantity,
    p.Price
FROM OrderDetails AS od
JOIN Products AS p
    ON od.ProductID = p.ProductID
""", connection)

order_lines46["line_value"] = (
    order_lines46["Quantity"] * order_lines46["Price"]
)

order_totals46 = (
    order_lines46.groupby("OrderID", as_index=False)
    .agg(total_line_value=("line_value","sum"))
)
print(order_lines46.head())
order_totals46.head()

### Key Takeaway

Calculate features at the grain where their inputs exist, then aggregate deliberately.

## Problem 47 — Northwind Multi-Table Join

### Problem

Combine orders, customers, order details, and products.

### Solution

The `Orders → OrderDetails` join changes grain from order to order line because one order can contain several products.

#### Implementation

In [ ]:
multi47 = pd.read_sql_query("""
SELECT
    o.OrderID,
    o.OrderDate,
    c.CustomerName,
    c.Country,
    p.ProductName,
    od.Quantity,
    p.Price
FROM Orders AS o
JOIN Customers AS c
    ON o.CustomerID = c.CustomerID
JOIN OrderDetails AS od
    ON o.OrderID = od.OrderID
JOIN Products AS p
    ON od.ProductID = p.ProductID
""", connection)
multi47.head()

### Key Takeaway

Restate row meaning after every structural join.

## Problem 48 — SQL Aggregation

### Problem

Count orders per customer.

### Solution

A left join plus grouping returns one row per customer, including customers with zero matching orders.

#### Implementation

In [ ]:
orders_per_customer48 = pd.read_sql_query("""
SELECT
    c.CustomerID,
    c.CustomerName,
    COUNT(o.OrderID) AS order_count
FROM Customers AS c
LEFT JOIN Orders AS o
    ON c.CustomerID = o.CustomerID
GROUP BY c.CustomerID, c.CustomerName
ORDER BY order_count DESC
""", connection)
orders_per_customer48.head(10)

### Key Takeaway

SQL `GROUP BY` and pandas `groupby` express analogous analytical structure at different execution layers.

## Problem 49 — SQL or pandas?

### Problem

Choose a suitable execution layer for six tasks.

### Solution

A reasonable mapping is: SQL for filtering 500M rows; SQL for joining large relational tables; pandas for interactive feature work on 50k rows; Python/NumPy/pandas for residuals already in memory; SQL or a distributed engine for billion-row aggregation; pandas for exploratory summaries after loading. Scale, location, and maintainability should drive the choice.

### Key Takeaway

Tool choice should follow the workload, not preference alone.

## Problem 50 — Preparing Predictors and Target

### Problem

Create predictor DataFrame `X` and target Series `y`.

### Solution

Multiple predictor columns remain a DataFrame; one selected target column is a Series.

#### Implementation

In [ ]:
df50 = pd.DataFrame({
    "age":[25,31,28,40,36],
    "income":[42000,51000,47000,68000,59000],
    "segment":["A","B","A","B","A"],
    "target":[0,1,0,1,1]
})
X50 = df50[["age","income","segment"]]
y50 = df50["target"]
print(type(X50), X50.shape)
print(type(y50), y50.shape)

### Key Takeaway

Keep the modelling interface explicit so the target cannot be accidentally included as a predictor.

## Problem 51 — One-Hot Encoding

### Problem

Encode the categorical `segment` variable.

### Solution

One-hot encoding replaces categories with indicator columns, producing a numerical representation.

#### Implementation

In [ ]:
X51 = pd.get_dummies(X50, columns=["segment"], dtype=int)
X51

### Key Takeaway

Encoding changes representation; it must be applied consistently to training and future data.

## Problem 52 — DataFrame to NumPy

### Problem

Convert encoded predictors to NumPy and compare.

### Solution

Shape is retained, while labels and index semantics are lost. The NumPy array is positional and homogeneous.

#### Implementation

In [ ]:
X52 = X51.to_numpy()
print("DataFrame:", X51.shape, X51.columns.tolist())
print("Array:", X52.shape, X52.dtype)
X52

### Key Takeaway

NumPy retains numerical structure, not DataFrame semantics.

## Problem 53 — Chained Assignment

### Problem

Rewrite a chained assignment with `.loc`.

### Solution

A single `.loc` operation makes both the row selection and destination column explicit.

#### Implementation

In [ ]:
df53 = df50.copy()
df53.loc[df53["income"] > 50000, "high_income"] = True
df53["high_income"] = df53["high_income"].fillna(False).astype(bool)
df53

### Key Takeaway

Prefer one explicit indexing operation for assignment.

## Problem 54 — Vectorization Versus Row Iteration

### Problem

Compute revenue using vectorised arithmetic.

### Solution

Vectorisation expresses the column relationship directly and avoids Python-level row iteration.

#### Implementation

In [ ]:
df54 = pd.DataFrame({
    "price":[10.0,15.0,8.0,20.0],
    "quantity":[3,2,5,4]
})
df54["revenue"] = df54["price"] * df54["quantity"]
df54

### Key Takeaway

When an operation is naturally column-wise, vectorisation is usually the clearest representation.

## Problem 55 — Diagnose a Merge Bug

### Problem

Explain unexpected growth from 10,000 to 12,500 rows after a supposed many-to-one merge.

### Solution

The likely cause is duplicate `CustomerID` values in the customer table, creating many-to-many matches. Check key uniqueness, inspect duplicates, and use `validate='many_to_one'`.

#### Implementation

In [ ]:
diagnostic = """
print(customers["CustomerID"].is_unique)

duplicates = customers.loc[
    customers["CustomerID"].duplicated(keep=False)
].sort_values("CustomerID")

print(duplicates)

result = orders.merge(
    customers,
    on="CustomerID",
    how="left",
    validate="many_to_one"
)
"""
print(diagnostic)

### Key Takeaway

Unexpected row growth is usually a key/cardinality problem.

## Problem 56 — Diagnose an Alignment Bug

### Problem

Assign a differently ordered Series to a DataFrame.

### Solution

Pandas aligns by index: `A` receives 20, `B` receives 30, and `C` receives 10.

#### Implementation

In [ ]:
df56 = pd.DataFrame({"value":[100,200,300]}, index=["A","B","C"])
adjustment56 = pd.Series([10,20,30], index=["C","A","B"])
df56["adjustment"] = adjustment56
df56

### Key Takeaway

Labels participate in assignment; physical order is not the default match rule.

## Problem 57 — Data Leakage Through Imputation

### Problem

Explain contamination from imputing before train/test split.

### Solution

The full-data mean contains information from held-out observations. Split first, fit the imputation statistic on training data, then apply the fitted transformation to validation/test data.

### Key Takeaway

Any preprocessing step that learns from data must respect the evaluation boundary.

## Problem 58 — Leakage Through Aggregation

### Problem

Explain why full-year spending can leak future information.

### Solution

If prediction occurs at the start of a month, later transactions are unavailable at prediction time. Features must be constructed only from information available up to each prediction cutoff.

### Key Takeaway

Feature validity is temporal as well as statistical.

## Problem 59 — Grain Reasoning

### Problem

Describe how to construct one row per customer from customer, order, and order-line tables.

### Solution

Aggregate order lines to order totals first if needed, merge those totals into orders, then group orders by customer to compute order count, quantity, and expenditure. Finally merge the customer-level features into the customer table. Track grain at every step.

### Key Takeaway

Feature engineering across relational tables is primarily a grain-management problem.

## Problem 60 — Integrated DataFrame Reasoning

### Problem

Explain a merge → datetime feature → groupby → aggregation → sort pipeline.

### Solution

`orders` starts at one row per order. The validated many-to-one merge adds customer attributes without changing grain. `order_month` derives a monthly period. Grouping by customer and month changes the grain to one row per customer-month. `nunique` counts distinct orders, `sum` aggregates amount, and sorting changes only presentation order.

#### Implementation

In [ ]:
orders60 = pd.DataFrame({
    "order_id":[1,2,3,4,5],
    "customer_id":[10,10,20,20,20],
    "order_date":pd.to_datetime([
        "2026-01-05","2026-01-20","2026-01-10",
        "2026-02-12","2026-02-20"
    ]),
    "amount":[100,150,80,120,60]
})
customers60 = pd.DataFrame({
    "customer_id":[10,20],
    "customer_name":["Alpha","Beta"]
})
result60 = (
    orders60
    .merge(customers60, on="customer_id", how="left", validate="many_to_one")
    .assign(order_month=lambda x: x["order_date"].dt.to_period("M"))
    .groupby(["customer_id","customer_name","order_month"], as_index=False)
    .agg(
        order_count=("order_id","nunique"),
        total_amount=("amount","sum")
    )
    .sort_values(["customer_id","order_month"])
)
result60

### Key Takeaway

Track grain, keys/cardinality, label alignment, and information boundaries throughout a pandas pipeline.

# Final Review

The exercises in this notebook move from basic DataFrame construction to the structural issues that determine whether a pandas workflow is analytically correct.

The most transferable concepts are:

- **grain:** what one row represents;
- **keys and cardinality:** how tables relate and whether a merge can multiply rows;
- **labels and alignment:** how pandas matches data across Series and DataFrames;
- **data types:** which operations have valid semantics;
- **missingness:** what is absent and how treatment choices affect analysis;
- **aggregation:** when row-level observations become group-level summaries;
- **reshaping:** how wide and long structures change representation and grain;
- **execution layer:** which work belongs naturally in SQL versus pandas;
- **modelling boundaries:** whether preprocessing and feature construction respect train/test and temporal availability constraints.

A pandas transformation is therefore not correct merely because it executes. It is correct when the resulting table still represents the intended population, grain, variables, and analytical information set.